# Model 3 — feature selection (scored at carrier handoff)

**Goal:** decide which of the 27 handoff-time candidate columns Model 3 should use.

**How to run:** upload `orders_clean.csv` to the Colab file panel (left sidebar), then *Runtime → Run all*. It takes about 10–20 minutes on a normal Colab CPU.

**Data rules (same split as the production Model 1):**

| Part | Purchase dates | Used here for |
|---|---|---|
| Training period | 2017-01-01 → 2018-03-31 | All feature selection (4 time-ordered folds inside it) |
| Calibration | April 2018 | One final sanity check of the chosen set |
| Test | May → Aug 2018 | **Not touched in this notebook** — kept for the final Model 3 evaluation |

**Method, in six steps:**
1. Single-column screen: how well each column alone separates late from on-time (ROC-AUC), and how many values are missing.
2. Redundancy check: pairs of numeric columns that carry the same information (|Spearman ρ| ≥ 0.8).
3. Baseline: XGBoost on all 27 columns, scored with 4 chronological folds.
4. Permutation importance on held-out folds: how much PR-AUC drops when a column is shuffled.
5. Recursive backward elimination: repeatedly drop the least useful column, re-measuring importance after every drop.
6. Pick the smallest set within 2% of the best mean PR-AUC (same rule as Model 1), check fold-by-fold stability, compare with simple alternatives, and save the list.

In [ ]:
# ---------------------------------------------------------------- setup
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, xgboost
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
try:
    display
except NameError:          # running outside Jupyter
    display = print

SEED = 42
rng_global = np.random.default_rng(SEED)
print("scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__)
print("Expected: scikit-learn 1.6.1 | xgboost 3.4.1 (the versions used for production Model 1)")

In [ ]:
# ---------------------------------------------------------------- load data
import os
for p in ["orders_clean.csv", "/content/orders_clean.csv", "/content/drive/MyDrive/orders_clean.csv"]:
    if os.path.exists(p):
        CLEAN_PATH = p
        break
else:
    raise FileNotFoundError("Upload orders_clean.csv to the Colab file panel first.")

clean = pd.read_csv(CLEAN_PATH, parse_dates=["purchase_ts"]).sort_values("purchase_ts").reset_index(drop=True)
print(f"Loaded {CLEAN_PATH}: {len(clean):,} rows x {clean.shape[1]} columns (expected 94,930 x 39)")
bad = clean["is_late"].isna() | clean["purchase_ts"].isna()
if bad.any():
    print(f"WARNING: {bad.sum()} row(s) have no is_late / purchase_ts — usually a half-finished upload "
          f"(the last line gets cut). Dropping them. If this is more than 1-2 rows, re-upload the file.")
    display(clean.loc[bad, ["order_id", "purchase_ts", "is_late"]].head())
    clean = clean[~bad].reset_index(drop=True)
clean["is_late"] = clean["is_late"].astype(int)

STAGE1 = ["promise_days", "purchase_hour", "purchase_weekday", "price", "freight", "n_items", "weight_g",
          "volume_cm3", "category", "customer_state", "seller_state", "same_state", "dist_km", "pay_type",
          "installments", "seller_prior_orders", "seller_late_rate_prior", "seller_overrun_rate_prior",
          "seller_handling_days_prior", "route_recent_days", "promise_buffer_route", "promise_buffer_state",
          "order_load_ratio"]
STAGE2 = ["approval_lag_h"]
STAGE3 = ["handling_days", "seller_overrun_days", "post_approval_days"]
CANDIDATES = STAGE1 + STAGE2 + STAGE3
CATEGORICAL = ["category", "customer_state", "seller_state", "pay_type"]
NUMERIC = [c for c in CANDIDATES if c not in CATEGORICAL]
STAGE_OF = {**{c: "1 purchase" for c in STAGE1}, **{c: "2 approval" for c in STAGE2}, **{c: "3 handoff" for c in STAGE3}}

missing = [c for c in CANDIDATES + ["order_id", "purchase_ts", "is_late"] if c not in clean.columns]
assert not missing, f"orders_clean.csv is missing {missing}"
assert len(CANDIDATES) == 27

P = clean.purchase_ts
TRAIN = clean[(P >= "2017-01-01") & (P < "2018-04-01")].copy()
CALIB = clean[(P >= "2018-04-01") & (P < "2018-05-01")].copy()
TEST_ROWS = int(((P >= "2018-05-01") & (P < "2018-09-01")).sum())
del P

print(f"Training period : {len(TRAIN):,} orders, {TRAIN.is_late.sum():,} late ({TRAIN.is_late.mean():.1%})")
print(f"April (calib)   : {len(CALIB):,} orders, {CALIB.is_late.sum():,} late ({CALIB.is_late.mean():.1%})")
print(f"Test (untouched): {TEST_ROWS:,} orders")

## Time-ordered folds

Each fold trains on everything before a cut-off, skips one month (an *embargo*, so orders still in transit at the cut-off cannot leak their outcome), then validates on the next two months. The folds deliberately include the Nov-2017 and Feb–Mar 2018 surge months.

In [ ]:
FOLDS = [  # (train before, validation start, validation end)
    ("2017-07-01", "2017-08-01", "2017-10-01"),
    ("2017-09-01", "2017-10-01", "2017-12-01"),
    ("2017-11-01", "2017-12-01", "2018-02-01"),
    ("2018-01-01", "2018-02-01", "2018-04-01"),
]
fold_idx = []
rows = []
for k, (tr_end, va_start, va_end) in enumerate(FOLDS, 1):
    tr = np.where(TRAIN.purchase_ts < tr_end)[0]
    va = np.where((TRAIN.purchase_ts >= va_start) & (TRAIN.purchase_ts < va_end))[0]
    fold_idx.append((tr, va))
    rows.append(dict(fold=k, train_until=tr_end, embargo_month=tr_end[:7], validate=f"{va_start} → {va_end}",
                     train_rows=len(tr), train_late=int(TRAIN.is_late.values[tr].sum()),
                     valid_rows=len(va), valid_late=int(TRAIN.is_late.values[va].sum()),
                     valid_late_rate=round(TRAIN.is_late.values[va].mean() * 100, 1)))
display(pd.DataFrame(rows))

## Step 1 — single-column screen

ROC-AUC of each column on its own, measured on the training period. 0.5 = no signal. Numbers below 0.5 are flipped ("direction-adjusted"), because a column whose *low* values mean "late" is just as useful. Categorical columns are scored by their training-period late rate per category (computed on the earlier part only, to avoid leaking).

In [ ]:
y_all = TRAIN.is_late.values

def single_auc(col):
    if col in CATEGORICAL:
        aucs = []
        for tr, va in fold_idx:   # category late rate learned on the fold's train part, scored on its valid part
            rate = pd.Series(y_all[tr]).groupby(TRAIN[col].astype(str).values[tr]).mean()
            s = TRAIN[col].astype(str).iloc[va].map(rate).fillna(y_all[tr].mean()).values
            aucs.append(roc_auc_score(y_all[va], s))
        return float(np.mean(aucs))
    x = TRAIN[col].values.astype(float)
    ok = ~np.isnan(x)
    return float(roc_auc_score(y_all[ok], x[ok]))

screen = []
for c in CANDIDATES:
    a = single_auc(c)
    screen.append(dict(feature=c, stage=STAGE_OF[c], type="categorical" if c in CATEGORICAL else "numeric",
                       auc_raw=round(a, 3), auc_adjusted=round(max(a, 1 - a), 3),
                       missing_pct=round(TRAIN[c].isna().mean() * 100, 1)))
screen = pd.DataFrame(screen).sort_values("auc_adjusted", ascending=False).reset_index(drop=True)
display(screen)

fig, ax = plt.subplots(figsize=(8, 7))
colors = screen.stage.map({"1 purchase": "#9aa5b1", "2 approval": "#e0a458", "3 handoff": "#2a9d8f"})
ax.barh(screen.feature[::-1], screen.auc_adjusted[::-1] - 0.5, left=0.5, color=colors[::-1])
ax.axvline(0.5, color="black", lw=0.8)
ax.set_xlabel("single-column ROC-AUC (direction-adjusted)")
ax.set_title("Step 1: signal of each column alone  (green = handoff, orange = approval, grey = purchase)")
plt.tight_layout(); plt.show()

## Step 2 — redundancy check

Pairs of numeric columns with |Spearman ρ| ≥ 0.8 carry largely the same information. We do **not** drop them blindly here: the elimination in step 5 re-measures importance after every drop, so when one of a pair goes, the other's importance rises and it is kept. This table is for the report and for checking the final list.

In [ ]:
rho = TRAIN[NUMERIC].corr(method="spearman")
pairs = [(a, b, round(rho.loc[a, b], 3)) for i, a in enumerate(NUMERIC) for b in NUMERIC[i + 1:]
         if abs(rho.loc[a, b]) >= 0.8]
redundant = pd.DataFrame(pairs, columns=["feature_a", "feature_b", "spearman_rho"])
display(redundant if len(redundant) else "no pairs with |rho| >= 0.8")

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(rho.abs(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(NUMERIC))); ax.set_xticklabels(NUMERIC, rotation=90, fontsize=8)
ax.set_yticks(range(len(NUMERIC))); ax.set_yticklabels(NUMERIC, fontsize=8)
plt.colorbar(im, label="|Spearman rho|"); ax.set_title("Step 2: overlap between numeric candidates")
plt.tight_layout(); plt.show()

## Model and cross-validation helpers

XGBoost with the same moderate settings as the production Model 1 (shallow trees, many small steps, class weight × 0.75). Tuning happens later, in the Model 3 training notebook — here we only need a fair, fixed yardstick for comparing column sets.

Categorical columns are mapped to integer codes learned on each fold's training part; categories never seen in training become −1. Missing numbers stay as NaN, which XGBoost handles natively.

In [ ]:
XGB_PARAMS = dict(n_estimators=600, max_depth=2, learning_rate=0.03, min_child_weight=10,
                  subsample=0.9, colsample_bytree=0.9, reg_lambda=10.0)
WEIGHT_MULTIPLIER = 0.75
TOP_CATEGORIES = 20

def encode(train_df, frames, feats):
    # Integer-encode categoricals using the training part only.
    out = [f[feats].copy() for f in [train_df] + list(frames)]
    if "category" in feats:
        top = train_df["category"].value_counts().head(TOP_CATEGORIES).index
        for X in out:
            X["category"] = X["category"].where(X["category"].isin(top), "other")
    for c in [c for c in CATEGORICAL if c in feats]:
        mapping = {v: i for i, v in enumerate(sorted(out[0][c].astype(str).unique()))}
        for X in out:
            X[c] = X[c].astype(str).map(mapping).fillna(-1).astype(int)
    return out

def fit(X, y):
    pw = (y == 0).sum() / max((y == 1).sum(), 1) * WEIGHT_MULTIPLIER
    m = XGBClassifier(**XGB_PARAMS, objective="binary:logistic", eval_metric="logloss", scale_pos_weight=pw,
                      tree_method="hist", random_state=SEED, n_jobs=-1, verbosity=0)
    return m.fit(X, y)

def recall_at_top(y, s, k=0.10):
    n = max(1, int(len(y) * k))
    return y[np.argsort(-s)[:n]].sum() / max(y.sum(), 1)

def cv_evaluate(feats, perm_repeats=0, seed=SEED):
    # Train/score on every fold. Optionally return permutation importance (PR-AUC drop) per feature per fold.
    rng = np.random.default_rng(seed)
    per_fold, imp = [], {f: [] for f in feats}
    for k, (tr, va) in enumerate(fold_idx, 1):
        Xtr, Xva = encode(TRAIN.iloc[tr], [TRAIN.iloc[va]], feats)
        ytr, yva = y_all[tr], y_all[va]
        model = fit(Xtr, ytr)
        s = model.predict_proba(Xva)[:, 1]
        base = average_precision_score(yva, s)
        per_fold.append(dict(fold=k, pr_auc=base, roc_auc=roc_auc_score(yva, s), recall_top10=recall_at_top(yva, s)))
        for f in (feats if perm_repeats else []):
            drops = []
            for _ in range(perm_repeats):
                Xp = Xva.copy()
                Xp[f] = rng.permutation(Xp[f].values)
                drops.append(base - average_precision_score(yva, model.predict_proba(Xp)[:, 1]))
            imp[f].append(float(np.mean(drops)))
    folds = pd.DataFrame(per_fold)
    return folds, (pd.DataFrame(imp, index=[f"fold{k}" for k in range(1, len(fold_idx) + 1)]).T if perm_repeats else None)

def summary(folds):
    return dict(pr_auc=folds.pr_auc.mean(), pr_auc_std=folds.pr_auc.std(), roc_auc=folds.roc_auc.mean(),
                recall_top10=folds.recall_top10.mean())

## Step 3 + 4 — baseline on all 27 columns and permutation importance

Permutation importance = how much the held-out PR-AUC falls when one column's values are shuffled (5 shuffles per column per fold). A column the model truly relies on causes a clear drop; a useless or fully duplicated column causes ~0 or a slightly negative change. Measuring it on held-out months, not on training rows, is what makes it trustworthy.

In [ ]:
t0 = time.time()
base_folds, base_imp = cv_evaluate(CANDIDATES, perm_repeats=5)
print(f"All 27 columns: {summary(base_folds)}   ({time.time() - t0:.0f}s)")
display(base_folds.round(4))

imp_table = pd.DataFrame({
    "stage": [STAGE_OF[f] for f in base_imp.index],
    "mean_pr_auc_drop": base_imp.mean(axis=1),
    "folds_positive": (base_imp > 0).sum(axis=1),
}, index=base_imp.index).join(base_imp).sort_values("mean_pr_auc_drop", ascending=False)
display(imp_table.round(4))

fig, ax = plt.subplots(figsize=(8, 7))
it = imp_table.iloc[::-1]
ax.barh(it.index, it.mean_pr_auc_drop, color=it.stage.map({"1 purchase": "#9aa5b1", "2 approval": "#e0a458", "3 handoff": "#2a9d8f"}))
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("mean PR-AUC drop when shuffled (held-out folds)")
ax.set_title("Step 4: permutation importance, all 27 columns")
plt.tight_layout(); plt.show()

## Step 5 — recursive backward elimination

Start from all 27. At each step: train on every fold, measure held-out permutation importance, drop the single least useful column, record the mean PR-AUC. Re-measuring after every drop matters: when one of two overlapping columns (e.g. `handling_days` / `post_approval_days`) is removed, the survivor's importance goes up and it is kept.

This is the slow cell (≈ 27 rounds × 4 folds).

In [ ]:
current = list(CANDIDATES)
path = []
t0 = time.time()
while True:
    folds, imp = cv_evaluate(current, perm_repeats=3 if len(current) > 1 else 0, seed=SEED + len(current))
    s = summary(folds)
    path.append(dict(n_features=len(current), **s, features=list(current),
                     fold_pr_auc=folds.pr_auc.round(4).tolist()))
    if len(current) == 1:
        break
    weakest = imp.mean(axis=1).idxmin()
    path[-1]["dropped_next"] = weakest
    print(f"{len(current):2d} cols | PR-AUC {s['pr_auc']:.4f} ± {s['pr_auc_std']:.4f} | ROC {s['roc_auc']:.4f} "
          f"| drop next: {weakest:28s} ({time.time() - t0:.0f}s)")
    current.remove(weakest)

path = pd.DataFrame(path)

## Step 6 — choose the set

Rule (same as Model 1): **the smallest set whose mean PR-AUC is within 2% of the best set's**. Fold-to-fold variation is large, so chasing the single highest number would just be fitting noise; a smaller set is also less to compute and explain in the stream.

In [ ]:
best = path.loc[path.pr_auc.idxmax()]
cutoff = best.pr_auc * 0.98
eligible = path[path.pr_auc >= cutoff]
chosen = eligible.loc[eligible.n_features.idxmin()]
SELECTED = list(chosen.features)

print(f"Best mean PR-AUC : {best.pr_auc:.4f} with {best.n_features} columns")
print(f"2% cut-off       : {cutoff:.4f}")
print(f"Chosen           : {chosen.n_features} columns, PR-AUC {chosen.pr_auc:.4f}, ROC {chosen.roc_auc:.4f}, "
      f"recall@top10% {chosen.recall_top10:.3f}")
print("\nSelected columns:")
for f in SELECTED:
    print(f"  {STAGE_OF[f]:11s} {f}")
print("\nDropped:", [f for f in CANDIDATES if f not in SELECTED])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(path.n_features, path.pr_auc, marker="o", color="#264653", label="mean PR-AUC (4 folds)")
ax.fill_between(path.n_features, path.pr_auc - path.pr_auc_std / 2, path.pr_auc + path.pr_auc_std / 2,
                color="#264653", alpha=0.12, label="± half a std across folds")
ax.axhline(cutoff, color="#e76f51", ls="--", lw=1, label="within 2% of best")
ax.axvline(chosen.n_features, color="#2a9d8f", ls=":", lw=1.5, label=f"chosen: {chosen.n_features} columns")
ax.invert_xaxis(); ax.set_xlabel("number of columns (eliminating left → right)"); ax.set_ylabel("PR-AUC")
ax.set_title("Step 5–6: backward elimination path"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

display(path[["n_features", "pr_auc", "pr_auc_std", "roc_auc", "recall_top10", "dropped_next", "fold_pr_auc"]].round(4))

### Stability check

A good column should help in **every** fold, not just one lucky period. Below: permutation importance of the chosen columns, fold by fold, re-measured with more shuffles.

In [ ]:
sel_folds, sel_imp = cv_evaluate(SELECTED, perm_repeats=10, seed=SEED + 999)
stab = pd.DataFrame({"stage": [STAGE_OF[f] for f in sel_imp.index],
                     "mean_pr_auc_drop": sel_imp.mean(axis=1),
                     "folds_positive": (sel_imp > 0).sum(axis=1)}, index=sel_imp.index).join(sel_imp)
display(stab.sort_values("mean_pr_auc_drop", ascending=False).round(4))
weak = stab[stab.folds_positive < 3].index.tolist()
print("Columns helping in fewer than 3 of 4 folds:", weak if weak else "none")

## Comparison with simple alternatives

Is the selection actually better than obvious hand-made sets? All scored with the same 4 folds.

* **all 27** — no selection
* **chosen** — result of this notebook
* **handoff only (4)** — just the approval/handoff timing columns
* **Model 1's 12 + handoff 4** — the production Model 1 columns plus the 4 new ones

In [ ]:
MODEL1_12 = ["promise_days", "purchase_hour", "price", "freight", "category", "customer_state", "dist_km",
             "seller_overrun_rate_prior", "seller_handling_days_prior", "route_recent_days",
             "promise_buffer_route", "promise_buffer_state"]
SETS = {
    "all 27": CANDIDATES,
    f"chosen ({len(SELECTED)})": SELECTED,
    "handoff only (4)": STAGE2 + STAGE3,
    "Model 1's 12 + handoff 4": MODEL1_12 + STAGE2 + STAGE3,
}
cmp_rows = []
for name, feats in SETS.items():
    f, _ = cv_evaluate(feats)
    cmp_rows.append(dict(set=name, n=len(feats), **summary(f), fold_pr_auc=f.pr_auc.round(4).tolist()))
comparison = pd.DataFrame(cmp_rows)
display(comparison.round(4))

## Final check on April 2018 (not the test set)

Train on the whole training period, score April — once per set. April was not used in any step above, so this is a fair out-of-sample check that the chosen set holds up. The May–Aug test set stays untouched until the final Model 3 evaluation.

In [ ]:
y_cal = CALIB.is_late.values
april = []
for name, feats in SETS.items():
    Xtr, Xca = encode(TRAIN, [CALIB], feats)
    s = fit(Xtr, y_all).predict_proba(Xca)[:, 1]
    april.append(dict(set=name, n=len(feats), april_pr_auc=average_precision_score(y_cal, s),
                      april_roc_auc=roc_auc_score(y_cal, s), april_recall_top10=recall_at_top(y_cal, s)))
april = pd.DataFrame(april)
display(april.round(4))
print(f"April base rate: {y_cal.mean():.2%}  ({y_cal.sum()} late of {len(y_cal):,})")
print("For reference, production Model 1 (purchase time) scored April PR-AUC ≈ 0.146, ROC-AUC ≈ 0.768.")

In [ ]:
# ---------------------------------------------------------------- save results
out = {
    "model": "model_3", "scored_at": "handed to carrier",
    "selected_features": SELECTED,
    "selected_categorical": [c for c in SELECTED if c in CATEGORICAL],
    "dropped_features": [f for f in CANDIDATES if f not in SELECTED],
    "selection_rule": "smallest set within 2% of best mean PR-AUC, recursive backward elimination "
                      "with held-out permutation importance, 4 chronological folds with 1-month embargo",
    "folds": FOLDS,
    "xgb_params": XGB_PARAMS, "weight_multiplier": WEIGHT_MULTIPLIER,
    "cv_selected": {k: float(chosen[k]) for k in ["pr_auc", "pr_auc_std", "roc_auc", "recall_top10"]},
    "cv_best": {"n_features": int(best.n_features), "pr_auc": float(best.pr_auc)},
    "data_split": {"train": ["2017-01-01", "2018-04-01"], "calibration": ["2018-04-01", "2018-05-01"],
                   "test_untouched": ["2018-05-01", "2018-09-01"]},
    "versions": {"scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__},
}
json.dump(out, open("model3_selected_features.json", "w"), indent=2, default=str)
screen.to_csv("model3_step1_single_column_screen.csv", index=False)
imp_table.to_csv("model3_step4_permutation_importance_all27.csv")
path.drop(columns=["features"]).assign(features=path.features.map(", ".join)).to_csv("model3_step5_elimination_path.csv", index=False)
comparison.merge(april, on=["set", "n"]).to_csv("model3_step6_set_comparison.csv", index=False)

import zipfile
files_out = ["model3_selected_features.json", "model3_step1_single_column_screen.csv",
             "model3_step4_permutation_importance_all27.csv", "model3_step5_elimination_path.csv",
             "model3_step6_set_comparison.csv"]
with zipfile.ZipFile("Model3_Feature_Selection.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files_out:
        z.write(f)
print("Saved Model3_Feature_Selection.zip:", files_out)
try:
    from google.colab import files
    files.download("Model3_Feature_Selection.zip")
except Exception:
    pass